# Previsão — Copa do Mundo 2026
## Modelo Final v6 — Features com Histórico de Copas

**Modelo selecionado:** Random Forest v6 (11 features, 216 amostras)

**Justificativa:** A v6 adicionou `media_gols_ultimas2_copas` e `fase_ultima_copa` às 9 features da v4. O Random Forest v6 apresentou melhoria estatisticamente significativa sobre o v4 (Wilcoxon p=0.031, MAE CV 0.4419 vs 0.4515), mantendo as 216 amostras de treino.

**Features v6:**
- 7 features originais (ciclo + últimos 15 jogos)
- `elo_medio_adv_ciclo` e `elo_medio_adv_ult15`
- `media_gols_ultimas2_copas` — histórico ofensivo nas Copas anteriores
- `fase_ultima_copa` — fase atingida na Copa imediatamente anterior

**Valor de mercado:** incluído na tabela final como dado contextual (não entra no modelo).

**Erro esperado:** ~0.44 gols por jogo

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')
np.random.seed(42)

df_features = pd.read_csv('../data/processed/features_completo_v6.csv')
df_raw      = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])

# Valor de mercado 2026
df_mercado_raw = pd.read_excel('../data/raw/football_team_values_2010_2026.xlsx',
                                sheet_name='Football Values')
df_mercado_raw.columns = ['copa', 'selecao', 'valor_mercado']
df_mercado_2026 = df_mercado_raw[df_mercado_raw['copa'] == 2026].copy()

print(f'Dataset treino (v6): {df_features.shape}')
print(f'Seleções com valor de mercado 2026: {len(df_mercado_2026)}')

## 2. ELO Histórico e Retreino

In [ ]:
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)
print(f'ELO calculado para {len(df)} jogos')

features = [
    'media_gols_marcados_ciclo', 'media_gols_sofridos_ciclo',
    'pct_vitorias_ciclo', 'total_jogos_ciclo',
    'media_gols_marcados_ult15', 'media_gols_sofridos_ult15',
    'pct_vitorias_ult15', 'elo_medio_adv_ciclo', 'elo_medio_adv_ult15',
    'media_gols_ultimas2_copas', 'fase_ultima_copa'
]

X_full = df_features[features]
y_full = df_features['media_gols_copa']

lr  = LinearRegression().fit(X_full, y_full)
rf  = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_full, y_full)
xgb = XGBRegressor(n_estimators=100, random_state=42).fit(X_full, y_full)

print(f'Modelos retreinados com {len(X_full)} amostras (1994-2022)')

## 3. Datas das Copas e Funções Auxiliares

In [ ]:
copas_datas = {
    1990: ('1990-06-08', '1990-07-08'),
    1994: ('1994-06-17', '1994-07-17'),
    1998: ('1998-06-10', '1998-07-12'),
    2002: ('2002-05-31', '2002-06-30'),
    2006: ('2006-06-09', '2006-07-09'),
    2010: ('2010-06-11', '2010-07-11'),
    2014: ('2014-06-12', '2014-07-13'),
    2018: ('2018-06-14', '2018-07-15'),
    2022: ('2022-11-20', '2022-12-18'),
}

def get_fase(df_copa, selecao):
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0: return 0
    n = len(jogos)
    if n == 3:   return 1
    elif n == 4: return 2
    elif n == 5: return 3
    elif n == 6: return 4
    elif n == 7:
        ultimo = jogos.sort_values('date').iloc[-1]
        ganhou = (ultimo['home_score'] > ultimo['away_score']) if ultimo['home_team'] == selecao                  else (ultimo['away_score'] > ultimo['home_score'])
        return 6 if ganhou else 5
    return min(n - 2, 6)

def get_media_gols_copa(df_copa, selecao):
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0: return np.nan
    return np.mean([
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in jogos.iterrows()
    ])

print('Funções definidas!')

## 4. Ciclo 2026 e Função de Features v6

In [ ]:
ciclo_2026 = df[
    (df['date'] >= '2022-12-20') &
    (df['date'] <= '2026-06-10') &
    (df['tournament'] != 'FIFA World Cup')
]
print(f'Jogos no ciclo 2026: {len(ciclo_2026)}')

def calcular_features_2026_v6(selecao, ciclo, min_jogos=10):
    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    if len(jogos) < min_jogos:
        raise ValueError(f'Apenas {len(jogos)} jogos')

    gm, gs, vit, elo_adv = [], [], [], []
    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gm.append(row['home_score']); gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score']); gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])

    gm, gs, vit = np.array(gm), np.array(gs), np.array(vit)
    elo_adv = np.array(elo_adv)

    ult15 = jogos.tail(15)
    gm15, gs15, vit15, ea15 = [], [], [], []
    for _, row in ult15.iterrows():
        if row['home_team'] == selecao:
            gm15.append(row['home_score']); gs15.append(row['away_score'])
            vit15.append(1 if row['home_score'] > row['away_score'] else 0)
            ea15.append(row['elo_away_antes'])
        else:
            gm15.append(row['away_score']); gs15.append(row['home_score'])
            vit15.append(1 if row['away_score'] > row['home_score'] else 0)
            ea15.append(row['elo_home_antes'])

    # Histórico Copa 2022 (última Copa antes de 2026)
    inicio_22, fim_22 = copas_datas[2022]
    copa_22 = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= inicio_22) & (df['date'] <= fim_22)
    ]
    inicio_18, fim_18 = copas_datas[2018]
    copa_18 = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= inicio_18) & (df['date'] <= fim_18)
    ]

    fase_22 = get_fase(copa_22, selecao)
    mg_22   = get_media_gols_copa(copa_22, selecao)
    mg_18   = get_media_gols_copa(copa_18, selecao)

    gols_hist = [g for g in [mg_22, mg_18] if not np.isnan(g)]
    media_hist = np.mean(gols_hist) if gols_hist else 0.0

    return {
        'media_gols_marcados_ciclo': gm.mean(),
        'media_gols_sofridos_ciclo': gs.mean(),
        'pct_vitorias_ciclo':        vit.mean(),
        'total_jogos_ciclo':         len(jogos),
        'media_gols_marcados_ult15': np.array(gm15).mean(),
        'media_gols_sofridos_ult15': np.array(gs15).mean(),
        'pct_vitorias_ult15':        np.array(vit15).mean(),
        'elo_medio_adv_ciclo':       elo_adv.mean(),
        'elo_medio_adv_ult15':       np.array(ea15).mean(),
        'media_gols_ultimas2_copas': media_hist,
        'fase_ultima_copa':          fase_22,
    }

print('Função v6 para 2026 definida!')

## 5. Seleções Classificadas

In [ ]:
selecoes_2026 = {
    'Argentina': 'CONMEBOL', 'Brazil': 'CONMEBOL', 'Colombia': 'CONMEBOL',
    'Ecuador': 'CONMEBOL', 'Paraguay': 'CONMEBOL', 'Uruguay': 'CONMEBOL',
    'England': 'UEFA', 'France': 'UEFA', 'Croatia': 'UEFA', 'Norway': 'UEFA',
    'Portugal': 'UEFA', 'Germany': 'UEFA', 'Netherlands': 'UEFA',
    'Austria': 'UEFA', 'Belgium': 'UEFA', 'Scotland': 'UEFA',
    'Spain': 'UEFA', 'Switzerland': 'UEFA', 'Sweden': 'UEFA',
    'Turkey': 'UEFA', 'Bosnia and Herzegovina': 'UEFA', 'Czech Republic': 'UEFA',
    'United States': 'CONCACAF', 'Canada': 'CONCACAF', 'Mexico': 'CONCACAF',
    'Panama': 'CONCACAF', 'Haiti': 'CONCACAF', 'Curaçao': 'CONCACAF',
    'Morocco': 'CAF', 'Senegal': 'CAF', 'Egypt': 'CAF', 'Ghana': 'CAF',
    'South Africa': 'CAF', 'Ivory Coast': 'CAF', 'Algeria': 'CAF',
    'Tunisia': 'CAF', 'Cape Verde': 'CAF', 'DR Congo': 'CAF',
    'Japan': 'AFC', 'South Korea': 'AFC', 'Iran': 'AFC', 'Australia': 'AFC',
    'Saudi Arabia': 'AFC', 'Iraq': 'AFC', 'Jordan': 'AFC',
    'Uzbekistan': 'AFC', 'Qatar': 'AFC',
    'New Zealand': 'OFC',
}
print(f'Total: {len(selecoes_2026)} seleções')

## 6. Calculando Features e Previsões

In [ ]:
nome_mercado_map = {
    'DR Congo': 'Democratic Republic of the Congo',
    'Turkey':   'Turkiye',
}

dados_2026 = []
sem_dados  = []

for selecao, conf in selecoes_2026.items():
    try:
        feat = calcular_features_2026_v6(selecao, ciclo_2026, min_jogos=10)
        feat['selecao']      = selecao
        feat['confederacao'] = conf

        nome_busca = nome_mercado_map.get(selecao, selecao)
        row_m = df_mercado_2026[
            df_mercado_2026['selecao'].str.lower() == nome_busca.lower()
        ]
        feat['valor_mercado_milhoes'] = float(row_m['valor_mercado'].values[0])             if len(row_m) > 0 else np.nan

        dados_2026.append(feat)
    except Exception as e:
        sem_dados.append(selecao)
        print(f'Sem dados: {selecao} — {e}')

df_2026 = pd.DataFrame(dados_2026)

X_2026 = df_2026[features]
df_2026['pred_lr']    = lr.predict(X_2026)
df_2026['pred_rf']    = rf.predict(X_2026)
df_2026['pred_xgb']   = xgb.predict(X_2026).clip(0)
df_2026['pred_media'] = df_2026[['pred_lr', 'pred_rf', 'pred_xgb']].mean(axis=1)
df_2026['divergencia'] = df_2026[['pred_lr', 'pred_rf', 'pred_xgb']].std(axis=1)

df_2026 = df_2026.sort_values('pred_media', ascending=False).reset_index(drop=True)

print(f'Previsões: {len(df_2026)} seleções')
print(f'Com valor de mercado: {df_2026["valor_mercado_milhoes"].notna().sum()}/{len(df_2026)}')
print('\nTop 20:')
print(df_2026[['selecao','confederacao','fase_ultima_copa','media_gols_ultimas2_copas','pred_media','valor_mercado_milhoes']]
      .head(20).to_string(index=False))

## 7. Visualização — Top 20

In [ ]:
cores_conf = {
    'CONMEBOL': '#1f77b4', 'UEFA': '#ff7f0e', 'CONCACAF': '#2ca02c',
    'CAF': '#d62728', 'AFC': '#9467bd', 'OFC': '#8c564b'
}

top20 = df_2026.head(20)
fig, ax = plt.subplots(figsize=(13, 9))

cores_barras = [cores_conf[c] for c in top20['confederacao'][::-1]]
bars = ax.barh(top20['selecao'][::-1], top20['pred_media'][::-1],
               color=cores_barras, edgecolor='black', alpha=0.85)

for bar, val in zip(bars, top20['pred_media'][::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)

for i, (_, row) in enumerate(top20[::-1].iterrows()):
    if not pd.isna(row['valor_mercado_milhoes']):
        ax.text(0.02, i, f'€{row["valor_mercado_milhoes"]:.0f}M',
                va='center', fontsize=7, color='white', fontweight='bold')

media_geral = df_2026['pred_media'].mean()
ax.axvline(media_geral, color='black', linestyle='--', alpha=0.4)

legend_conf = [mpatches.Patch(color=c, label=conf) for conf, c in cores_conf.items()]
legend_conf.append(plt.Line2D([0],[0], color='black', linestyle='--',
                               label=f'Média: {media_geral:.2f}'))
ax.legend(handles=legend_conf, loc='lower right', fontsize=8)
ax.set_xlabel('Média de Gols por Jogo (prevista)')
ax.set_title(
    'Previsão de Média de Gols — Copa do Mundo 2026\n'
    'Top 20 | Features v6 (histórico Copas + ELO adversários) | Ensemble 3 modelos',
    fontsize=11
)
ax.set_xlim(0, top20['pred_media'].max() + 0.4)
plt.tight_layout()
plt.savefig('../article/figures/previsao_2026_top20_final.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlação: Previsão vs Valor de Mercado

In [ ]:
df_com_m = df_2026.dropna(subset=['valor_mercado_milhoes'])
corr = df_com_m['valor_mercado_milhoes'].corr(df_com_m['pred_media'])

fig, ax = plt.subplots(figsize=(10, 6))
cores = [cores_conf[c] for c in df_com_m['confederacao']]
ax.scatter(df_com_m['valor_mercado_milhoes'], df_com_m['pred_media'],
           c=cores, s=80, edgecolors='black', alpha=0.8)

for _, row in df_com_m.iterrows():
    ax.annotate(row['selecao'],
                (row['valor_mercado_milhoes'], row['pred_media']),
                fontsize=7, ha='left', va='bottom')

z = np.polyfit(df_com_m['valor_mercado_milhoes'], df_com_m['pred_media'], 1)
x_l = np.linspace(df_com_m['valor_mercado_milhoes'].min(),
                  df_com_m['valor_mercado_milhoes'].max(), 100)
ax.plot(x_l, np.poly1d(z)(x_l), 'r--', alpha=0.6, label=f'Correlação: {corr:.2f}')

legend_conf = [mpatches.Patch(color=c, label=conf) for conf, c in cores_conf.items()]
legend_conf.append(plt.Line2D([0],[0], color='r', linestyle='--',
                               label=f'Correlação: {corr:.2f}'))
ax.legend(handles=legend_conf, fontsize=8)
ax.set_xlabel('Valor de Mercado (€ milhões)')
ax.set_ylabel('Média de Gols Prevista')
ax.set_title('Consistência do Modelo v6: Previsão vs Valor de Mercado 2026')
plt.tight_layout()
plt.savefig('../article/figures/previsao_vs_mercado_2026.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlação Pearson: {corr:.4f}')

## 9. Visualização por Confederação

In [ ]:
media_conf = df_2026.groupby('confederacao')['pred_media'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores = [cores_conf[c] for c in media_conf.index]
axes[0].bar(media_conf.index, media_conf.values, color=cores, edgecolor='black', alpha=0.85)
for i, (conf, val) in enumerate(media_conf.items()):
    axes[0].text(i, val + 0.01, f'{val:.2f}', ha='center', fontsize=9)
axes[0].set_title('Média de Gols Prevista por Confederação — Copa 2026')
axes[0].set_ylabel('Média de Gols por Jogo')

sns.boxplot(data=df_2026, x='confederacao', y='pred_media',
            order=media_conf.index.tolist(), palette=cores_conf,
            ax=axes[1], legend=False)
axes[1].set_title('Distribuição das Previsões por Confederação')
axes[1].set_ylabel('Média de Gols por Jogo (prevista)')
axes[1].set_xlabel('')
plt.tight_layout()
plt.savefig('../article/figures/previsao_2026_confederacoes_final.png', dpi=150, bbox_inches='tight')
plt.show()

print('Média por confederação:')
print(media_conf.round(3).to_string())

## 10. Salvamento

In [ ]:
cols = [
    'selecao', 'confederacao', 'total_jogos_ciclo',
    'media_gols_marcados_ciclo', 'media_gols_marcados_ult15',
    'elo_medio_adv_ciclo', 'fase_ultima_copa',
    'media_gols_ultimas2_copas', 'valor_mercado_milhoes',
    'pred_lr', 'pred_rf', 'pred_xgb', 'pred_media', 'divergencia'
]

df_2026[cols].to_csv('../data/processed/previsao_2026_final.csv', index=False)
df_2026[cols].to_csv('../article/tables/previsao_2026_final.csv', index=False)
print('Salvo!')
print(f'Total: {len(df_2026)} seleções')
print(f'\nResumo (ensemble):')
print(df_2026['pred_media'].describe().round(3))

## 11. Conclusões

**Modelo final:** Random Forest v6 — 11 features, 216 amostras, MAE CV 0.4419.

**Evolução do projeto:**
- v1 baseline → v4 com ELO adversários → v6 com histórico Copas
- Única melhoria estatisticamente significativa: RF v6 (Wilcoxon p=0.031)

**Valor de mercado:** contextual na tabela — correlação 0.54 com as previsões confirma consistência do modelo.

**Erro esperado:** ~0.44 gols por jogo por seleção.